# Aula 04 — Chunking Strategies Evaluation

Pipeline: PDF → Markdown → chunking (10 strategies) → embeddings → JSON.

This notebook builds the pipeline incrementally. This first section
defines the shared data contracts and configuration used by every later
stage, so the chunking, embedding, and JSON-export steps can be built
independently and still agree on the same data shapes.

In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

### 1. Embedding Model Configuration

Kept as a single configurable constant so all 10 tests run with the exact
same embedding model, per the assignment's comparability requirement.

In [2]:
EMBEDDING_MODEL = "openai/text-embedding-3-small"  # via OpenRouter

### 2. Output Structure

```
results/
├── <document_id>/
│   ├── markdown/<document_id>.md
│   ├── test_01/chunks_embeddings.json
│   ├── ...
│   └── test_10/chunks_embeddings.json
└── summary.json
```

In [3]:
RESULTS_DIR = Path("results")


def markdown_path(document_id: str) -> Path:
    """Path to the intermediate Markdown file for a document."""
    return RESULTS_DIR / document_id / "markdown" / f"{document_id}.md"


def chunks_path(document_id: str, test_id: int) -> Path:
    """Path to the chunks+embeddings JSON for one document/test combination."""
    return RESULTS_DIR / document_id / f"test_{test_id:02d}" / "chunks_embeddings.json"


def summary_path() -> Path:
    """Path to the overall experiment summary JSON."""
    return RESULTS_DIR / "summary.json"

### 3. Chunk Record — JSON Schema for a Single Chunk

Matches the schema requested in the assignment (section 10).

In [4]:
@dataclass
class ChunkMetadata:
    """Optional structural metadata attached to a chunk, when available.

    Not every chunking strategy can populate every field (e.g. only the
    Markdown-header strategy knows `heading`/`heading_level`). Fields that
    stay `None` are dropped when serializing, instead of writing `null`
    noise into every chunk.
    """

    page: int | None = None
    section: str | None = None
    heading: str | None = None
    heading_level: int | None = None


@dataclass
class ChunkRecord:
    """A single chunk plus its embedding, ready to be serialized to JSON."""

    chunk_id: str
    document_id: str
    document_name: str
    test_id: int
    strategy: str
    chunk_size: int | None
    chunk_overlap: int | None
    text: str
    embedding: list[float]
    metadata: ChunkMetadata = field(default_factory=ChunkMetadata)

    def to_dict(self) -> dict[str, Any]:
        """Convert to a plain dict, ready for `json.dump`."""
        metadata_dict = {k: v for k, v in vars(self.metadata).items() if v is not None}
        return {
            "chunk_id": self.chunk_id,
            "document_id": self.document_id,
            "document_name": self.document_name,
            "test_id": self.test_id,
            "strategy": self.strategy,
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "text": self.text,
            "embedding": self.embedding,
            "metadata": metadata_dict,
        }


def make_chunk_id(document_id: str, test_id: int, index: int) -> str:
    """Build a chunk_id in the format used across the assignment's examples.

    Example: make_chunk_id("doc01", 5, 1) -> "doc01_test05_chunk001"
    """
    return f"{document_id}_test{test_id:02d}_chunk{index:03d}"

### 4. Test Configuration — the 10 Chunking Experiments

`strategy` groups tests that share an implementation (e.g. tests 1-4 are
all plain fixed-size splits; only `chunk_size` changes). The chunker for
each strategy is implemented in the next pipeline stage — this section
only declares *what* to run, not *how*.

In [5]:
StrategyName = Literal[
    "fixed",
    "fixed_with_overlap",
    "paragraph",
    "sentence_grouped",
    "recursive",
    "markdown",
]


@dataclass
class TestConfig:
    """Configuration for a single chunking experiment (one row of the test matrix)."""

    test_id: int
    strategy: StrategyName
    label: str
    params: dict[str, Any]


TEST_CONFIGS: list[TestConfig] = [
    TestConfig(1, "fixed", "Fixed 200 chars, no overlap", {"chunk_size": 200, "chunk_overlap": 0}),
    TestConfig(2, "fixed", "Fixed 500 chars, no overlap", {"chunk_size": 500, "chunk_overlap": 0}),
    TestConfig(3, "fixed", "Fixed 1000 chars, no overlap", {"chunk_size": 1000, "chunk_overlap": 0}),
    TestConfig(4, "fixed", "Fixed 2000 chars, no overlap", {"chunk_size": 2000, "chunk_overlap": 0}),
    TestConfig(5, "fixed_with_overlap", "Fixed 500 chars, overlap 50", {"chunk_size": 500, "chunk_overlap": 50}),
    TestConfig(6, "fixed_with_overlap", "Fixed 500 chars, overlap 200", {"chunk_size": 500, "chunk_overlap": 200}),
    TestConfig(7, "paragraph", "Paragraph-based split", {}),
    TestConfig(8, "sentence_grouped", "Sentences grouped in 3", {"sentences_per_chunk": 3}),
    # chunk_size/overlap are a starting point — the assignment asks us to
    # justify the chosen parameters, so these are meant to be tuned (and
    # the justification written up) in the chunking step, not fixed here.
    TestConfig(9, "recursive", "Recursive character split", {"chunk_size": 1000, "chunk_overlap": 100}),
    TestConfig(10, "markdown", "Markdown header-based split", {}),
]

### 5. Sanity Check

Quick check that the test matrix matches the assignment's table before
wiring up the chunkers.

In [6]:
print(f"Embedding model: {EMBEDDING_MODEL}\n")
print(f"{'ID':<4}{'Strategy':<20}{'Label':<32}Params")
for test in TEST_CONFIGS:
    print(f"{test.test_id:<4}{test.strategy:<20}{test.label:<32}{test.params}")

Embedding model: openai/text-embedding-3-small

ID  Strategy            Label                           Params
1   fixed               Fixed 200 chars, no overlap     {'chunk_size': 200, 'chunk_overlap': 0}
2   fixed               Fixed 500 chars, no overlap     {'chunk_size': 500, 'chunk_overlap': 0}
3   fixed               Fixed 1000 chars, no overlap    {'chunk_size': 1000, 'chunk_overlap': 0}
4   fixed               Fixed 2000 chars, no overlap    {'chunk_size': 2000, 'chunk_overlap': 0}
5   fixed_with_overlap  Fixed 500 chars, overlap 50     {'chunk_size': 500, 'chunk_overlap': 50}
6   fixed_with_overlap  Fixed 500 chars, overlap 200    {'chunk_size': 500, 'chunk_overlap': 200}
7   paragraph           Paragraph-based split           {}
8   sentence_grouped    Sentences grouped in 3          {'sentences_per_chunk': 3}
9   recursive           Recursive character split       {'chunk_size': 1000, 'chunk_overlap': 100}
10  markdown            Markdown header-based split     {}


## Step 2 — Chunking Strategies

Implements the 10 chunking experiments using LangChain's text splitters.
Each strategy function takes the document's Markdown text and a
`TestConfig`, and returns a list of `(chunk_text, metadata)` pairs — kept
separate from `ChunkRecord` here, since the chunk_id/document fields are
only known once we know which document is being processed.

Strategies 1-6 (fixed-size, with and without overlap) and 9 (recursive)
map directly onto `CharacterTextSplitter` / `RecursiveCharacterTextSplitter`.
Strategy 8 (group of 3 sentences) has no equivalent built into LangChain,
so it's implemented manually with a lightweight regex sentence splitter,
as the assignment allows when no appropriate splitter exists.

In [7]:
!pip install -q langchain-text-splitters

In [8]:
import re

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

### 2.1 Strategies 1-6 — Fixed Size (With and Without Overlap)

`CharacterTextSplitter` with an empty separator splits on raw character
count, ignoring word or sentence boundaries — this is what makes it a
useful baseline against the structure-aware strategies below (paragraph,
recursive, Markdown).

In [9]:
def split_fixed(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    """Split text into fixed-size chunks, ignoring word/sentence boundaries.

    Used for tests 1-6 (plain fixed-size and fixed-size with overlap).
    """
    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        strip_whitespace=False,
    )
    return splitter.split_text(text)

### 2.2 Strategy 7 — Paragraph

Each paragraph (a block separated by a blank line) becomes exactly one
chunk. Setting `chunk_size=1` on `RecursiveCharacterTextSplitter` with a
single separator (`"\n\n"`) is a deliberate trick: since no two paragraphs
can ever fit under a size of 1, the splitter never merges them, and since
there is no fallback separator, it never re-splits a paragraph either —
the net effect is "one paragraph in, one chunk out", using the library's
own splitter instead of a manual `str.split("\n\n")`.

In [10]:
def split_paragraphs(text: str) -> list[str]:
    """Split text into paragraphs (blocks separated by a blank line).

    Used for test 7. See markdown note above for why chunk_size=1 is used.
    """
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n"],
        chunk_size=1,
        chunk_overlap=0,
    )
    chunks = splitter.split_text(text)
    return [c.strip() for c in chunks if c.strip()]

### 2.3 Strategy 8 — Sentences Grouped in 3

No LangChain splitter groups a fixed number of sentences per chunk, so
this strategy is implemented manually with a regex-based sentence
boundary detector (splits after `.`, `!`, or `?` followed by whitespace).
It is not a full NLP-grade tokenizer (it won't handle abbreviations like
"Dr." correctly), but it is adequate for grouping sentences into chunks.

In [11]:
def split_into_sentences(text: str) -> list[str]:
    """Split text into sentences using a punctuation-based heuristic."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sentences if s]


def split_sentence_grouped(text: str, sentences_per_chunk: int = 3) -> list[str]:
    """Group sentences into chunks of `sentences_per_chunk` each.

    Used for test 8.
    """
    sentences = split_into_sentences(text)
    return [
        " ".join(sentences[i:i + sentences_per_chunk])
        for i in range(0, len(sentences), sentences_per_chunk)
    ]

### 2.4 Strategy 9 — Recursive

Uses `RecursiveCharacterTextSplitter` with its default separator
hierarchy (paragraphs → lines → words → characters), falling back to a
finer-grained separator only when a chunk doesn't fit — this is the
"composite" strategy the assignment refers to.

In [12]:
def split_recursive(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    """Split text recursively, preferring paragraph/line/word boundaries.

    Used for test 9. chunk_size=1000, chunk_overlap=100 were chosen as a
    middle ground: large enough to preserve multi-sentence context (unlike
    tests 1-2), small enough to keep chunks focused (unlike test 4), with
    a modest overlap to avoid losing context at chunk boundaries.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)

### 2.5 Strategy 10 — Markdown Headers

Uses `MarkdownHeaderTextSplitter` to split on `#`/`##`/`###` headings,
preserving the heading hierarchy as metadata for each chunk (mapped onto
`ChunkMetadata.heading` / `heading_level` / `section` below).

In [13]:
MARKDOWN_HEADERS = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]


def split_markdown(text: str) -> list[tuple[str, ChunkMetadata]]:
    """Split text on Markdown headings, keeping heading info as metadata.

    Used for test 10. Returns (chunk_text, metadata) pairs, since this is
    the only strategy that produces heading-aware metadata directly.
    """
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=MARKDOWN_HEADERS,
        strip_headers=False,
    )
    documents = splitter.split_text(text)

    results = []
    for doc in documents:
        # The deepest header present (e.g. "Header 2" over "Header 1") is
        # the most specific heading for this chunk.
        header_keys = [key for key in ("Header 3", "Header 2", "Header 1") if key in doc.metadata]
        deepest_key = header_keys[0] if header_keys else None

        metadata = ChunkMetadata(
            heading=doc.metadata.get(deepest_key) if deepest_key else None,
            heading_level=int(deepest_key[-1]) if deepest_key else None,
            section=doc.metadata.get("Header 1"),
        )
        results.append((doc.page_content.strip(), metadata))

    return results

### 2.6 Strategy Dispatch

Ties every `TestConfig` to its chunking function, and normalizes all
outputs to the same `(chunk_text, metadata)` shape so the pipeline's next
stage (embeddings + JSON export) doesn't need to know which strategy
produced each chunk.

In [14]:
def run_chunking_strategy(text: str, test: TestConfig) -> list[tuple[str, ChunkMetadata]]:
    """Run the chunking strategy described by `test` against `text`.

    Returns a list of (chunk_text, metadata) pairs, regardless of which
    underlying strategy produced them. Empty or whitespace-only chunks are
    dropped — they carry no content to embed and some embedding APIs
    reject empty input outright.
    """
    if test.strategy in ("fixed", "fixed_with_overlap"):
        chunks = split_fixed(text, test.params["chunk_size"], test.params["chunk_overlap"])
        result = [(chunk, ChunkMetadata()) for chunk in chunks]

    elif test.strategy == "paragraph":
        chunks = split_paragraphs(text)
        result = [(chunk, ChunkMetadata()) for chunk in chunks]

    elif test.strategy == "sentence_grouped":
        chunks = split_sentence_grouped(text, test.params["sentences_per_chunk"])
        result = [(chunk, ChunkMetadata()) for chunk in chunks]

    elif test.strategy == "recursive":
        chunks = split_recursive(text, test.params["chunk_size"], test.params["chunk_overlap"])
        result = [(chunk, ChunkMetadata()) for chunk in chunks]

    elif test.strategy == "markdown":
        result = split_markdown(text)

    else:
        raise ValueError(f"Unknown strategy: {test.strategy}")

    return [(chunk, metadata) for chunk, metadata in result if chunk.strip()]

## Step 3 — Embeddings and JSON Export

Generates an embedding for every chunk produced in Step 2, and writes the
results to disk following the `results/` structure defined in Step 1
(one `chunks_embeddings.json` per document/test combination, plus a
single `summary.json` with comparative stats across all 10 strategies).

### 3.1 API Setup

Same pattern used in the Aula 03 notebook: load the API key from Colab
Secrets, and fetch embeddings in batches while tracking token usage.

In [15]:
from google.colab import userdata
import requests

API_KEY = userdata.get("OPENROUTER_API_KEY")

if not API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to Colab's secrets (key icon in the sidebar) before continuing."
    )

In [16]:
class InsufficientCreditsError(RuntimeError):
    """Raised when the embeddings API rejects a request for exceeding
    the account's remaining credits/quota (HTTP 402). Retrying is
    pointless until the account is topped up or a different embedding
    provider is configured — callers should stop, not retry.
    """


def fetch_embeddings(texts: list[str]) -> tuple[list[list[float]], int]:
    """Fetch embeddings for a batch of texts from the OpenRouter API.

    Args:
        texts: List of strings to embed.

    Returns:
        A tuple of (embeddings, tokens_used):
            - embeddings: one vector per input text, in the same order as
              `texts`.
            - tokens_used: total tokens billed for this request, as
              reported by the API's `usage` field.

    Raises:
        requests.HTTPError: If the API request fails.
    """
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": EMBEDDING_MODEL,
            "input": texts,
        },
    )
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        if response.status_code == 402:
            # Retrying won't help here — the account is out of credits/quota.
            # Raised as its own type so callers can stop immediately instead
            # of uselessly repeating this failure for every remaining test.
            raise InsufficientCreditsError(
                f"Insufficient OpenRouter credits/quota: {response.text}"
            ) from error
        # The API's error body usually explains *why* (e.g. an oversized
        # chunk exceeding the model's token limit) — surface it instead of
        # letting it get lost in a bare "400 Bad Request".
        raise requests.HTTPError(
            f"{error} | Response body: {response.text}", response=response
        ) from error

    payload = response.json()
    sorted_items = sorted(payload["data"], key=lambda item: item["index"])
    embeddings = [item["embedding"] for item in sorted_items]
    tokens_used = payload.get("usage", {}).get("total_tokens", 0)

    return embeddings, tokens_used


def generate_embeddings_in_batches(
    texts: list[str], batch_size: int = 50
) -> tuple[list[list[float] | None], int]:
    """Fetch embeddings for many texts, sending them in fixed-size batches.

    If a whole batch is rejected (e.g. one oversized chunk exceeding the
    model's token limit), it is retried one text at a time instead of
    discarding every chunk in that batch — only the actual offender ends
    up as `None` in the result, marking it as skipped.

    Args:
        texts: Texts to embed.
        batch_size: Maximum number of texts sent per API call.

    Returns:
        A tuple of (embeddings, total_tokens_used). `embeddings` has one
        entry per input text, in order; an entry is `None` if that
        specific chunk could not be embedded.

    Raises:
        InsufficientCreditsError: Propagated immediately, without retrying
            — the account needs credits before any further call can succeed.
    """
    all_embeddings: list[list[float] | None] = []
    total_tokens = 0

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        try:
            batch_embeddings, batch_tokens = fetch_embeddings(batch)
        except InsufficientCreditsError:
            raise
        except requests.HTTPError:
            batch_embeddings = []
            batch_tokens = 0
            for offset, single_text in enumerate(batch):
                try:
                    single_embedding, single_tokens = fetch_embeddings([single_text])
                    batch_embeddings.append(single_embedding[0])
                    batch_tokens += single_tokens
                except InsufficientCreditsError:
                    raise
                except requests.HTTPError as single_error:
                    print(f"    chunk {start + offset} skipped (embedding failed): {single_error}")
                    batch_embeddings.append(None)

        all_embeddings.extend(batch_embeddings)
        total_tokens += batch_tokens

    return all_embeddings, total_tokens

### 3.2 Process One Document/Test Combination

Runs a single chunking strategy against a document, embeds every chunk,
and returns the finished `ChunkRecord` list plus the tokens spent.

In [17]:
def process_document_test(
    document_id: str,
    document_name: str,
    text: str,
    test: TestConfig,
) -> tuple[list[ChunkRecord], int]:
    """Chunk a document with one strategy and embed every resulting chunk.

    Returns:
        A tuple of (chunk_records, tokens_used).
    """
    chunks = run_chunking_strategy(text, test)
    chunk_texts = [chunk_text for chunk_text, _ in chunks]

    embeddings, tokens_used = generate_embeddings_in_batches(chunk_texts)

    records = []
    skipped = 0
    for index, ((chunk_text, metadata), embedding) in enumerate(zip(chunks, embeddings), start=1):
        if embedding is None:
            # This specific chunk failed to embed (e.g. exceeded the
            # model's token limit) — drop just this one, not the whole
            # test's results.
            skipped += 1
            continue
        records.append(ChunkRecord(
            chunk_id=make_chunk_id(document_id, test.test_id, index),
            document_id=document_id,
            document_name=document_name,
            test_id=test.test_id,
            strategy=test.strategy,
            chunk_size=test.params.get("chunk_size"),
            chunk_overlap=test.params.get("chunk_overlap"),
            text=chunk_text,
            embedding=embedding,
            metadata=metadata,
        ))

    if skipped:
        print(f"    {skipped}/{len(chunks)} chunks skipped in this test")

    return records, tokens_used

### 3.3 Save Chunks and Build Summary Stats

In [18]:
import json


def save_chunks(document_id: str, test_id: int, records: list[ChunkRecord]) -> None:
    """Write a document/test's chunk records to their JSON file."""
    path = chunks_path(document_id, test_id)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump([record.to_dict() for record in records], f, ensure_ascii=False, indent=2)


def measure_overlap(chunk_a: str, chunk_b: str) -> int:
    """Length of the longest suffix of `chunk_a` that matches a prefix of `chunk_b`.

    Measures overlap empirically from the actual chunk text, instead of
    only trusting the configured `chunk_overlap` parameter — splitters can
    adjust the exact cut point to respect a separator, so the real overlap
    may differ slightly from what was requested.
    """
    limit = min(len(chunk_a), len(chunk_b))
    for length in range(limit, 0, -1):
        if chunk_a[-length:] == chunk_b[:length]:
            return length
    return 0


def summarize_experiment(
    test: TestConfig, records: list[ChunkRecord], tokens_used: int
) -> dict[str, Any]:
    """Build one experiment's entry for summary.json.

    Includes the overlap statistics and token count requested by the
    assignment for tests 1-6 (number of overlapping chunks, overlap
    percentage, tokens spent), alongside the size statistics already
    tracked for every test.
    """
    sizes = [len(record.text) for record in records]
    avg_size = sum(sizes) / len(sizes) if sizes else 0
    embedding_dimension = len(records[0].embedding) if records else 0

    overlap_lengths = [
        measure_overlap(records[i].text, records[i + 1].text)
        for i in range(len(records) - 1)
    ]
    overlapping_chunks = sum(1 for length in overlap_lengths if length > 0)
    avg_overlap_chars = sum(overlap_lengths) / len(overlap_lengths) if overlap_lengths else 0
    overlap_percentage = (avg_overlap_chars / avg_size * 100) if avg_size else 0

    return {
        "test_id": test.test_id,
        "strategy": test.strategy,
        "chunk_size": test.params.get("chunk_size"),
        "chunk_overlap": test.params.get("chunk_overlap"),
        "num_chunks": len(records),
        "avg_chunk_size": round(avg_size, 1),
        "min_chunk_size": min(sizes, default=0),
        "max_chunk_size": max(sizes, default=0),
        "overlapping_chunks": overlapping_chunks,
        "avg_overlap_chars": round(avg_overlap_chars, 1),
        "overlap_percentage": round(overlap_percentage, 1),
        "embedding_dimension": embedding_dimension,
        "tokens_used": tokens_used,
    }

### 3.4 Orchestration — Run All 10 Strategies for a Document

Processes every `TestConfig` against one document, saving each result and
collecting the comparative stats for `summary.json`.

In [19]:
def process_document(document_id: str, document_name: str, text: str) -> dict[str, Any]:
    """Run all 10 chunking strategies against one document.

    Saves each strategy's chunks to its own JSON file, and returns this
    document's entry for the overall summary.json. A single test failing
    (e.g. a chunk too large for the embedding model's token limit) is
    logged and recorded as an error entry, instead of aborting the run
    for the whole document — real documents in the corpus have wildly
    different structures, and one edge case shouldn't cost the results
    already computed for the other 9 strategies, or for the remaining
    documents.
    """
    experiments = []
    total_tokens = 0

    for test in TEST_CONFIGS:
        try:
            records, tokens_used = process_document_test(document_id, document_name, text, test)
            save_chunks(document_id, test.test_id, records)
            experiments.append(summarize_experiment(test, records, tokens_used))
            total_tokens += tokens_used
            print(f"[{document_id}] test {test.test_id:02d} ({test.strategy}): "
                  f"{len(records)} chunks, {tokens_used} tokens")
        except InsufficientCreditsError:
            # Not a per-test problem — every remaining call will fail the
            # same way, so stop this document (and the whole run) instead
            # of logging the identical failure dozens more times.
            raise
        except Exception as error:
            print(f"[{document_id}] test {test.test_id:02d} ({test.strategy}) FAILED: {error}")
            experiments.append({
                "test_id": test.test_id,
                "strategy": test.strategy,
                "chunk_size": test.params.get("chunk_size"),
                "chunk_overlap": test.params.get("chunk_overlap"),
                "error": str(error),
            })

    print(f"[{document_id}] total tokens spent: {total_tokens}\n")
    return {"document": document_name, "experiments": experiments}

## Step 4 — PDF to Markdown Extraction

Converts the real PDFs from the shared Drive folder into structured
Markdown, using [Docling](https://github.com/docling-project/docling) —
the same tool used in Aula 02. This step feeds Step 3 (chunking +
embeddings), so it must run before the full multi-document run below.

In [20]:
!pip install -q docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.5/729.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.8/286.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8

### 4.1 Converter Configuration

OCR is disabled: every PDF in the Drive folder is a native-text document
(academic papers), so OCR is unnecessary and, as noted in the Aula 02
README, can cause memory errors (`std::bad_alloc`) on high-resolution
pages. If a scanned PDF is added later, this flag should be flipped.

### 4.1a Disable torch.compile (Colab CPU compatibility)

Docling's layout model uses `torch.compile` internally for speed. On some Colab CPU runtimes, the underlying C++ compiler fails to build the optimized kernel (`CppCompileError`), which crashes the conversion. Disabling Dynamo/Inductor forces eager-mode execution instead — slower, but avoids depending on the runtime's C++ toolchain.

In [21]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
torch._dynamo.config.suppress_errors = True

In [22]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False

pdf_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

### 4.2 Convert a Single PDF

In [23]:
def convert_pdf_to_markdown(pdf_path: Path, document_id: str) -> str:
    """Convert one PDF to structured Markdown and save it under `markdown_path()`.

    Args:
        pdf_path: Path to the source PDF.
        document_id: Identifier used to name the output file and, later,
            every chunk derived from this document.

    Returns:
        The extracted Markdown text.
    """
    result = pdf_converter.convert(str(pdf_path))
    markdown_text = result.document.export_to_markdown()

    output_path = markdown_path(document_id)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(markdown_text, encoding="utf-8")

    return markdown_text

### 4.3 Upload and Convert All PDFs

Select every file from `AULA_04/pdfs/` in the upload dialog (multi-file
selection is supported).

In [24]:
from google.colab import files

uploaded_pdfs = files.upload()

Saving scaling_laws_llm.pdf to scaling_laws_llm.pdf
Saving retrieval_augmented_generation.pdf to retrieval_augmented_generation.pdf
Saving lora_low_rank_adaptation.pdf to lora_low_rank_adaptation.pdf
Saving llama_foundation_models.pdf to llama_foundation_models.pdf
Saving instruct_gpt.pdf to instruct_gpt.pdf
Saving gpt4_technical_report.pdf to gpt4_technical_report.pdf
Saving gpt3_language_models.pdf to gpt3_language_models.pdf
Saving bert_pretraining.pdf to bert_pretraining.pdf
Saving attention_is_all_you_need.pdf to attention_is_all_you_need.pdf
Saving twitter_algoritmo.pdf to twitter_algoritmo.pdf
Saving escrita_academica_ia.pdf to escrita_academica_ia.pdf
Saving bioetica_e_ia.pdf to bioetica_e_ia.pdf


In [25]:
# document_id is the filename without extension (e.g. "attention_is_all_you_need")
def clean_document_id(filename: str) -> str:
    """Derive a clean document_id from an uploaded filename.

    Colab's upload widget appends " (1)", " (2)", etc. to the filename
    when a file with the same name already exists in the session's disk
    (e.g. from re-running this cell) — this strips that suffix so
    re-uploads don't create duplicate, inconsistent document IDs.
    """
    return re.sub(r" \(\d+\)$", "", Path(filename).stem)


pdf_documents: dict[str, str] = {
    clean_document_id(name): name for name in uploaded_pdfs.keys()
}

markdown_texts: dict[str, str] = {}

for document_id, pdf_name in pdf_documents.items():
    markdown_texts[document_id] = convert_pdf_to_markdown(Path(pdf_name), document_id)
    print(f"{document_id}: {len(markdown_texts[document_id]):,} characters extracted")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

scaling_laws_llm: 99,708 characters extracted
retrieval_augmented_generation: 71,960 characters extracted
lora_low_rank_adaptation: 99,729 characters extracted
llama_foundation_models: 105,322 characters extracted
instruct_gpt: 220,005 characters extracted
gpt4_technical_report: 288,442 characters extracted
gpt3_language_models: 333,644 characters extracted
bert_pretraining: 70,235 characters extracted
attention_is_all_you_need: 48,957 characters extracted
twitter_algoritmo: 54,412 characters extracted
escrita_academica_ia: 42,148 characters extracted
bioetica_e_ia: 51,213 characters extracted


### 4.4 Structural Element Check — Images and Tables

The assignment specifically asks how images and tables survive the PDF
→ Markdown conversion. Docling represents tables as proper Markdown
pipe tables and images as `<!-- image -->` placeholder comments (the
image itself is not embedded in the text), so both are counted here as
a quick, reproducible signal — inspect the printed examples manually to
confirm before writing the final report.

In [26]:
def inspect_structural_elements(markdown_text: str) -> dict[str, int]:
    """Count structural markers relevant to the assignment's analysis
    questions about how tables and images are represented after conversion.
    """
    return {
        "markdown_tables": len(re.findall(r"^\|.+\|$\n\|[-: |]+\|", markdown_text, flags=re.MULTILINE)),
        "image_placeholders": markdown_text.count("<!-- image -->"),
        "headings": len(re.findall(r"^#{1,6}\s", markdown_text, flags=re.MULTILINE)),
    }


for document_id, text in markdown_texts.items():
    stats = inspect_structural_elements(text)
    print(f"{document_id}: {stats}")

scaling_laws_llm: {'markdown_tables': 9, 'image_placeholders': 24, 'headings': 52}
retrieval_augmented_generation: {'markdown_tables': 10, 'image_placeholders': 4, 'headings': 35}
lora_low_rank_adaptation: {'markdown_tables': 33, 'image_placeholders': 8, 'headings': 40}
llama_foundation_models: {'markdown_tables': 34, 'image_placeholders': 2, 'headings': 54}
instruct_gpt: {'markdown_tables': 45, 'image_placeholders': 24, 'headings': 132}
gpt4_technical_report: {'markdown_tables': 13, 'image_placeholders': 29, 'headings': 213}
gpt3_language_models: {'markdown_tables': 86, 'image_placeholders': 34, 'headings': 55}
bert_pretraining: {'markdown_tables': 9, 'image_placeholders': 5, 'headings': 33}
attention_is_all_you_need: {'markdown_tables': 12, 'image_placeholders': 6, 'headings': 27}
twitter_algoritmo: {'markdown_tables': 0, 'image_placeholders': 23, 'headings': 21}
escrita_academica_ia: {'markdown_tables': 4, 'image_placeholders': 8, 'headings': 20}
bioetica_e_ia: {'markdown_tables': 0

## Step 5 — Full Run: All Documents, All 10 Strategies

Now that every PDF has been converted, this replaces the 3-document
quick validation from Step 3.5 with the complete run requested by the
assignment: all documents, all 10 chunking strategies, embeddings, and
the final `summary.json`.

In [ ]:
def save_summary(summary_data: list[dict[str, Any]]) -> None:
    """Write the current summary_data to summary.json.

    Called after every document (not just once at the end), so a crash
    partway through the run doesn't discard results — and tokens — already
    spent on earlier documents.
    """
    summary_path().parent.mkdir(parents=True, exist_ok=True)
    with open(summary_path(), "w", encoding="utf-8") as f:
        json.dump(summary_data, f, ensure_ascii=False, indent=2)


def is_already_processed(document_id: str) -> bool:
    """Check whether a document's last strategy (test 10) was already saved.

    Lets a re-run of this cell skip documents fully processed in a
    previous attempt, instead of re-embedding (and re-billing) them.
    """
    return chunks_path(document_id, TEST_CONFIGS[-1].test_id).exists()


# Resume from a previous partial run, if summary.json already exists
if summary_path().exists():
    with open(summary_path(), "r", encoding="utf-8") as f:
        summary_data = json.load(f)
    processed_ids = {entry["document"] for entry in summary_data}
else:
    summary_data = []
    processed_ids = set()

for document_id, text in markdown_texts.items():
    document_name = pdf_documents[document_id]

    if document_name in processed_ids or is_already_processed(document_id):
        print(f"[{document_id}] already processed — skipping")
        continue

    try:
        summary_data.append(process_document(document_id, document_name, text))
    except InsufficientCreditsError as error:
        # Every remaining call would fail identically — stop the whole run
        # instead of looping through the rest of the documents uselessly.
        save_summary(summary_data)
        print(f"\n🛑 Stopped at [{document_id}]: {error}")
        print("Add credits at https://openrouter.ai/settings/credits, or "
              "switch EMBEDDING_MODEL to a free alternative (e.g. Hugging "
              "Face's Inference API), then re-run this cell — already "
              "completed documents will be skipped automatically.")
        break
    except Exception as error:
        # A different whole-document failure (e.g. a network error)
        # shouldn't cost the documents already processed before it.
        print(f"[{document_id}] DOCUMENT FAILED: {error}")
        summary_data.append({"document": document_name, "error": str(error)})

    save_summary(summary_data)

print(f"\nSummary written to {summary_path()}")
print(f"Documents processed: {len(summary_data)}")